# Mew3D Cloud Server (Colab free T4)
Full-quality Hunyuan3D **mesh + texture** as a temporary API for the local Mew3D project.

**Use:** Runtime > Change runtime type > **T4 GPU**, then run the 3 cells in order (first run ~10 min: installs + ~12GB models). Cell 3 prints a public URL like `https://xxxx.gradio.live` - open it in a browser, or call it from the project with `gradio_client`. Keep the tab open while using it; Colab kills idle sessions after ~90 min.

In [ ]:
# 1) Install (compiles the CUDA rasterizer - Colab has nvcc, takes a few minutes)
!pip -q install hy3dgen==2.0.2 gradio pymeshlab rembg onnxruntime
!git clone -q --depth 1 https://github.com/Tencent-Hunyuan/Hunyuan3D-2.git /content/h3d
!cd /content/h3d/hy3dgen/texgen/custom_rasterizer && pip -q install .
!cd /content/h3d/hy3dgen/texgen/differentiable_renderer && pip -q install . || echo 'optional compile skipped'
print('SETUP DONE')

In [ ]:
# 2) Load models (mini-turbo shape + full paint; T4 16GB fits both)
import torch
from hy3dgen.shapegen import (Hunyuan3DDiTFlowMatchingPipeline, FloaterRemover,
                              DegenerateFaceRemover, FaceReducer)
from hy3dgen.texgen import Hunyuan3DPaintPipeline

shape_pipe = Hunyuan3DDiTFlowMatchingPipeline.from_pretrained(
    'tencent/Hunyuan3D-2mini', subfolder='hunyuan3d-dit-v2-mini-turbo', use_safetensors=True)
try:
    shape_pipe.enable_flashvdm(mc_algo='mc')
except Exception as e:
    print('flashvdm off:', e)
paint_pipe = Hunyuan3DPaintPipeline.from_pretrained('tencent/Hunyuan3D-2')
print('MODELS LOADED')

In [ ]:
# 3) Serve: prints a public https://...gradio.live URL
import gradio as gr, tempfile, torch
from hy3dgen.rembg import BackgroundRemover

def generate(image, do_texture=True, steps=5, octree=380, seed=1234):
    if image.mode != 'RGBA':
        image = BackgroundRemover()(image.convert('RGB'))  # auto bg removal for raw photos
    mesh = shape_pipe(image=image, num_inference_steps=int(steps), guidance_scale=5.0,
                      octree_resolution=int(octree),
                      generator=torch.Generator().manual_seed(int(seed)))[0]
    if isinstance(mesh, list):
        mesh = mesh[0]
    mesh = FloaterRemover()(mesh)
    mesh = DegenerateFaceRemover()(mesh)
    mesh = FaceReducer()(mesh, max_facenum=40000)
    if do_texture:
        mesh = paint_pipe(mesh, image=image)
    path = tempfile.NamedTemporaryFile(suffix='.glb', delete=False).name
    mesh.export(path)
    return path

gr.Interface(
    generate,
    inputs=[gr.Image(type='pil', image_mode='RGBA', label='input image'),
            gr.Checkbox(True, label='texture'), gr.Number(5, label='steps'),
            gr.Number(380, label='octree resolution'), gr.Number(1234, label='seed')],
    outputs=gr.File(label='GLB'),
    title='Mew3D Cloud Backend (Hunyuan3D mesh + texture)',
).launch(share=True)